<a href="https://colab.research.google.com/github/MathMachado/DSWP/blob/master/Notebooks/UFG_Metodos_de_Boosting_de_alta_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Boosting

São métodos que representam o estado da arte para dados estruturados.

Diferente do *Bagging* (visto em Random Forest), onde os modelos são criados de forma independente e paralela, o **Boosting** é um processo **iterativo e sequencial**, onde cada novo modelo tenta corrigir os erros dos modelos anteriores[cite: 3268, 3270].

---

## 1. Fundamentos do Boosting de Alta Performance

### Gradient Boosting Machine (GBM)
O Gradient Boosting trata a construção do modelo como um problema de otimização numérica.
* **A Ideia:** Em vez de apenas reponderar observações (como no AdaBoost), o GBM treina cada nova árvore para prever os **resíduos** (o erro) do modelo anterior[cite: 3041, 3132].
* **Gradiente:** Ele utiliza o gradiente da função de perda para determinar a direção do ajuste[cite: 2379, 2389]. É, essencialmente, uma descida de gradiente no espaço das funções.



### XGBoost (eXtreme Gradient Boosting)
O XGBoost é uma implementação otimizada do GBM que revolucionou o aprendizado de máquina devido à sua eficiência e precisão.
* **Expansão de Taylor:** Enquanto o GBM tradicional usa apenas a primeira derivada, o XGBoost utiliza uma aproximação de **segunda ordem** (Taylor) da função de perda para otimizar a convergência[cite: 2369, 2381, 2382].
* **Divisão de Nós:** Utiliza algoritmos sofisticados para encontrar divisões (*splits*) ideais, lidando nativamente com dados esparsos e valores ausentes.

### LightGBM
Desenvolvido pela Microsoft, o LightGBM é focado em velocidade e eficiência em grandes volumes de dados.
* **Leaf-wise vs. Level-wise:** Enquanto o XGBoost cresce por níveis, o LightGBM cresce por **folhas** (*leaf-wise*), escolhendo a folha que reduz mais a perda global, o que resulta em convergência mais rápida.
* **GOSS e EFB:** Utiliza técnicas como amostragem baseada em gradiente (GOSS) e empacotamento de variáveis (*Bundling*) para reduzir drasticamente o custo computacional[cite: 3058, 3061].

---

## 2. Funções de Perda e Regularização

Em concursos, é comum cobrarem como esses modelos controlam a complexidade para evitar o *overfitting*.

* **Funções de Perda:**
    * **Regressão:** Mínimos quadrados (MSE) ou Erro Absoluto (MAE)[cite: 209, 246, 248].
    * **Classificação:** *Log-Loss* (Deviance binária), muito similar à verossimilhança da regressão logística[cite: 2320, 2322, 2948].
* **Regularização:**
    * **Penalidades L1 (Lasso) e L2 (Ridge):** Aplicadas diretamente na estrutura da árvore para penalizar nós complexos e folhas com pesos muito altos[cite: 1239, 1245].
    * **Shrinkage (Learning Rate):** Reduz o impacto de cada árvore individual, forçando o modelo a aprender mais devagar, o que geralmente melhora a generalização.

---

## 3. Tuning e Validação

O "ajuste fino" é mais complexo no Boosting do que no Random Forest.

* **Parâmetros Críticos:**
    * **Taxa de Aprendizado (learning rate):** Geralmente entre 0.01 e 0.3.
    * **Número de Árvores:** Exige cuidado; se for muito alto, causa *overfitting*.
    * **Profundidade das Árvores:** Normalmente usam-se árvores rasas (*weak learners*) para evitar a memorização dos ruídos.
* **Validação:** O método padrão é a **Validação Cruzada (K-fold)**[cite: 651, 931].
    * **Early Stopping:** Uma técnica vital onde o treinamento para se o erro em um conjunto de validação parar de cair por um número determinado de rodadas[cite: 499, 512].



---

## 4. Aplicações em Grandes Volumes e Limitações

### Big Data
* **Escalabilidade:** O LightGBM e o XGBoost suportam processamento distribuído (Hadoop, Spark) e uso de GPU.
* **Histogram-based learning:** Em vez de testar todos os valores possíveis para divisões, os dados são agrupados em *bins* (histogramas), acelerando o cálculo imensamente[cite: 1917, 1941, 1945].

### Limitações Teóricas
* **Sensibilidade a Ruído:** Por focarem intensamente nos erros (resíduos), os modelos de Boosting podem tentar "explicar" ruídos e *outliers*, levando ao sobreajuste se não houver regularização estrita[cite: 1269, 1270].
* **Custo Computacional:** Embora otimizados, são sequenciais por natureza, o que os torna mais lentos para treinar do que o Random Forest (que é paralelo).
* **Dificuldade de Ajuste:** Possuem muitos hiperparâmetros interdependentes, exigindo maior expertise do estatístico.

---

### Exemplo
o **Gradient Boosting** busca o mínimo da função de perda através de passos sucessivos no gradiente.

Para aprofundar na derivação matemática do gradiente da função de perda logística, utilizaremos os fundamentos teóricos presentes nos manuais de **Modelos Lineares Generalizados (MLG)** fornecidos, pois a **Log-Loss** utilizada no Boosting é, essencialmente, o negativo da log-verossimilhança da distribuição Bernoulli[cite: 1852, 2320].

Como você está se preparando para um concurso de docência, a derivação abaixo segue o rigor formal necessário.

---

### 1. Definição da Função de Perda (Log-Loss)
Em problemas de classificação binária, a variável resposta $Y_i$ assume valores $\{0, 1\}$[cite: 1773, 1842]. O modelo estima a probabilidade $\pi_i = P(Y_i = 1)$ através da função de ligação **logit**[cite: 1853, 1857]:
$$\eta_i = \log \left( \frac{\pi_i}{1 - \pi_i} \right) \implies \pi_i = \frac{1}{1 + e^{-\eta_i}}$$ [cite: 1864, 1866]

A função de perda logística (ou deviance binária) para uma observação é o negativo do logaritmo da função de massa probabilística da Bernoulli[cite: 1775, 2320]:
$$L(y_i, \eta_i) = -[y_i \log(\pi_i) + (1 - y_i) \log(1 - \pi_i)]$$ [cite: 2320, 2328]

### 2. Derivação do Gradiente em relação ao Preditor Linear
No contexto de **Gradient Boosting**, precisamos do gradiente da perda em relação ao ajuste atual do modelo ($\eta_i$). Para facilitar a derivação, reescrevemos $L$ em termos de $\eta_i$:
1. Note que $\log(\pi_i) = \log\left(\frac{e^{\eta_i}}{1 + e^{\eta_i}}\right) = \eta_i - \log(1 + e^{\eta_i})$[cite: 2322].
2. Note que $\log(1 - \pi_i) = \log\left(\frac{1}{1 + e^{\eta_i}}\right) = -\log(1 + e^{\eta_i})$.

Substituindo na função de perda:
$$L(y_i, \eta_i) = -[y_i (\eta_i - \log(1 + e^{\eta_i})) + (1 - y_i)(-\log(1 + e^{\eta_i}))]$$
$$L(y_i, \eta_i) = -y_i \eta_i + \log(1 + e^{\eta_i})$$ [cite: 2322, 2328]

Agora, aplicamos a derivada parcial (gradiente) em relação a $\eta_i$:
$$\frac{\partial L(y_i, \eta_i)}{\partial \eta_i} = -y_i + \frac{1}{1 + e^{\eta_i}} \cdot e^{\eta_i}$$
$$\frac{\partial L(y_i, \eta_i)}{\partial \eta_i} = -y_i + \pi_i$$

**Resultado do Gradiente:**
$$g_i = \pi_i - y_i$$ [cite: 2359, 2450]

> **Interpretação Teórica:** O gradiente nada mais é do que o **resíduo** (valor predito menos valor observado)[cite: 2359]. No Gradient Boosting, cada nova árvore será treinada para prever o negativo desse gradiente ($y_i - \pi_i$), que é a direção de descida que minimiza o erro[cite: 2470].



---

### 3. Derivação da Segunda Derivada (Hessiana)
Para algoritmos de alta performance como o **XGBoost**, utiliza-se a expansão de Taylor de segunda ordem, exigindo a Hessiana ($h_i$)[cite: 601, 2452]:
$$h_i = \frac{\partial^2 L(y_i, \eta_i)}{\partial \eta_i^2} = \frac{\partial}{\partial \eta_i} (\pi_i - y_i)$$

Como $\pi_i = (1 + e^{-\eta_i})^{-1}$, usamos a regra da cadeia:
$$\frac{\partial \pi_i}{\partial \eta_i} = \frac{e^{-\eta_i}}{(1 + e^{-\eta_i})^2} = \frac{1}{1 + e^{-\eta_i}} \cdot \frac{e^{-\eta_i}}{1 + e^{-\eta_i}}$$
$$h_i = \pi_i (1 - \pi_i)$$ [cite: 2453, 2460]

**Resultado da Hessiana:**
$$h_i = \pi_i (1 - \pi_i)$$ [cite: 2460, 2465]

> **Conexão com MLG:** Note que a Hessiana ($h_i$) é exatamente a **Variância** da distribuição Bernoulli[cite: 1773, 1827]. Isso explica por que, no algoritmo de estimação IWLS para MLGs, os pesos da matriz diagonal $W$ são dados por $\pi_i(1-\pi_i)$[cite: 2460, 2467].

### Resumo para Questões de Prova:
* **Gradiente ($g_i$):** Direção do erro ($\pi_i - y_i$). É o que o Gradient Boosting comum utiliza como "pseudo-resíduo"[cite: 2905, 2913].
* **Hessiana ($h_i$):** Curvatura da perda ($\pi_i(1 - \pi_i)$). Algoritmos como XGBoost a utilizam para ajustar o tamanho do passo de forma mais precisa, equivalente a um passo de Newton[cite: 609, 2369].

Este paralelo entre a otimização de modelos de Machine Learning e a estimação por Máxima Verossimilhança em Estatística clássica é um tema recorrente em provas de alto nível.


Para entender como a Hessiana é aplicada no cálculo do ganho em algoritmos de boosting de alta performance (como o **XGBoost**), precisamos analisar a função objetivo penalizada. Esse é o ponto onde o aprendizado de máquina se funde com a **Otimização Numérica** e a **Expansão de Taylor**.

Aqui está a derivação formal de como a Hessiana ($h_i$) e o Gradiente ($g_i$) determinam a estrutura da árvore.

---

### 1. A Função Objetivo Aproximada
Seja $L$ a função de perda (como a Log-Loss derivada anteriormente). No passo $t$, queremos adicionar uma árvore $f_t$ que minimize a perda total somada a um termo de regularização $\Omega(f_t)$:
$$Obj^{(t)} = \sum_{i=1}^n L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$

Aplicando a **Série de Taylor de 2ª ordem** em torno do ajuste anterior $\hat{y}_i^{(t-1)}$:
$$Obj^{(t)} \approx \sum_{i=1}^n \left[ L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$

Onde:
* **$g_i$ (Gradiente):** $\partial L / \partial \hat{y}^{(t-1)}$ (Resíduo).
* **$h_i$ (Hessiana):** $\partial^2 L / \partial (\hat{y}^{(t-1)})^2$ (Curvatura/Variância).



---

### 2. O Peso Ótimo das Folhas
Uma árvore de decisão atribui um valor constante $w_j$ a cada folha $j$. Removendo os termos constantes e definindo a estrutura da árvore, o objetivo para uma folha específica $j$ (que contém um conjunto de índices de observações $I_j$) é minimizar:
$$Obj_j = \left( \sum_{i \in I_j} g_i \right) w_j + \frac{1}{2} \left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2$$

Derivando em relação a $w_j$ e igualando a zero, encontramos o **peso ótimo da folha**:
$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

**Insight Teórico:** Note a semelhança com a estatística clássica: o peso é uma razão entre a soma dos resíduos e a soma das variâncias (mais a regularização $\lambda$). Sem a Hessiana, o algoritmo não saberia o "tamanho do passo" ideal para cada folha.

---

### 3. A Função de Pontuação (Score) e o Ganho
Ao substituir o peso ótimo $w_j^*$ de volta na função objetivo, obtemos a "qualidade" de uma estrutura de árvore (Score):
$$Score = -\frac{1}{2} \sum_{j=1}^T \frac{\left( \sum_{i \in I_j} g_i \right)^2}{\sum_{i \in I_j} h_i + \lambda} + \gamma T$$

Para decidir onde dividir um nó em dois (Esquerda $L$ e Direita $R$), calculamos o **Ganho**:
$$Ganho = \frac{1}{2} \left[ \underbrace{\frac{(\sum g_L)^2}{\sum h_L + \lambda}}_{\text{Score Esquerda}} + \underbrace{\frac{(\sum g_R)^2}{\sum h_R + \lambda}}_{\text{Score Direita}} - \underbrace{\frac{(\sum g_{Total})^2}{\sum h_{Total} + \lambda}}_{\text{Score sem divisão}} \right] - \gamma$$



---

### Resumo para a prova

1.  **Tratamento da Curvatura:** Ao usar a Hessiana, o XGBoost executa um passo de **Newton-Raphson** em vez de uma descida de gradiente simples[cite: 2368, 2369, 2415]. Isso acelera imensamente a convergência.
2.  **Regularização Adaptativa:** A Hessiana ($h_i$) atua como um peso natural. Se a perda tem alta curvatura (grande $h_i$), o algoritmo é mais cauteloso no ajuste do peso $w_j$.
3.  **Conexão com GLM:** Na perda logística, $h_i = \pi_i(1-\pi_i)$. Isso significa que observações próximas à fronteira de decisão ($p \approx 0.5$) têm maior Hessiana e, portanto, maior influência no cálculo do ganho, de forma análoga ao **Método de Fisher** em MLGs[cite: 2415, 2460].
4.  **O parâmetro $\gamma$:** No cálculo do ganho, o $\gamma$ atua como um custo de complexidade para adicionar uma nova folha. Se o ganho matemático for menor que $\gamma$, a árvore não cresce (poda automática).

Deseja praticar com um exemplo numérico simples de cálculo de ganho para uma folha com três observações?